# 🧬 GIADA Task 16 — conferma embedded congelata
Riproduciamo transizioni `deterministic_test` con il teacher NEURON autentico a 642 segmenti. La LUT-513 selezionata nella 15c corre come **shadow** sui voltaggi osservati nel medesimo replay; non modifica il teacher. Il midpoint usa il campione V successivo del sottopasso, quindi resta teacher-forced. Nessun riaddestramento, nessuna sostituzione causale.


In [ ]:
from pathlib import Path
import base64,hashlib,json,os,shutil,subprocess,sys,zipfile
from IPython.display import Javascript,display
WORK=Path('/kaggle/working/giada_roadmap_task16');REPO=WORK/'giada';TEACHER=WORK/'neuron_as_deep_net'
assert not WORK.exists(),f'Workspace già presente: {WORK}. Avvia una sessione pulita.'
WORK.mkdir(parents=True)
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert (REPO/'src/giada_teacher/roadmap_embedded_frozen_confirmation.py').is_file(),'Revisione senza Task 16.'
sys.path.insert(0,str(REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]:del sys.modules[name]
from src.giada_teacher import ExtractedGateFormula
from src.giada_teacher.roadmap_embedded_interface_bridge import verified_task5_any
from src.giada_teacher.roadmap_embedded_frozen_confirmation import FrozenEmbeddedConfig,verified_15c,run_frozen_embedded_confirmation
print({'revision':REVISION})


## 🧰 Preflight del teacher NEURON
Il replay nativo richiede NEURON 8.2.7 e i meccanismi canonici compilati. Prepariamo entrambe le dipendenze **prima** di verificare l'HDF5 da 6 GiB; non modifichiamo i sorgenti `.mod`.


In [ ]:
try:
 import neuron
except ModuleNotFoundError:
 subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
 import neuron
assert neuron.__version__=='8.2.7',f'NEURON {neuron.__version__} non corrisponde alla versione registrata 8.2.7.'
nrnivmodl=shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl')
assert Path(nrnivmodl).is_file(),'nrnivmodl non trovato dopo l’installazione di NEURON.'
assert shutil.which('gcc') is not None,'gcc non disponibile: impossibile compilare i meccanismi.'
simulation_dir=TEACHER/'L5PC_NEURON_simulation'
if not list(simulation_dir.rglob('libnrnmech.so')):
 subprocess.run([nrnivmodl,'mods'],cwd=simulation_dir,check=True)
assert list(simulation_dir.rglob('libnrnmech.so')),'Compilazione dei meccanismi NEURON fallita.'
print({'neuron':neuron.__version__,'compiled_mechanisms':'ready'})


## 📁 Input necessari
Aggiungi il dataset `hayflow-targeted-transition-dataset-v1-1-base` **con gli snapshot nativi**, `giada_primitive_scaling_laws.zip` (Task 5), e `giada_roadmap_task15c_embedded_interface_bridge.zip`. Il dataset può essere una cartella Kaggle o `archive.zip`. Se il mount ha un nome diverso, imposta `GIADA_TARGETED_DATASET`, `GIADA_TASK5_ARTIFACT`, `GIADA_TASK15C_ARTIFACT`. Non servono i vecchi dataset storici.


In [ ]:
INPUT=Path('/kaggle/input')
def find_verified(env,predicates,verify):
 override=os.environ.get(env);paths=[Path(override).expanduser()] if override else []
 if INPUT.is_dir():paths += predicates()
 for attempt,path in enumerate(dict.fromkeys(paths)):
  if not path.exists():continue
  try:verify(path,attempt);return path.resolve()
  except (ValueError,RuntimeError,FileNotFoundError,OSError,zipfile.BadZipFile):continue
 raise AssertionError(f'{env}: artefatto esatto non trovato negli Input Kaggle.')
TASK15C=find_verified('GIADA_TASK15C_ARTIFACT',lambda:[p for p in INPUT.rglob('*.zip') if 'task15c' in str(p).lower() or 'interface_bridge' in str(p).lower()]+[p.parent for p in INPUT.rglob('final_report.json') if 'task15c' in str(p).lower()],lambda p,_:verified_15c(p))
TASK5=find_verified('GIADA_TASK5_ARTIFACT',lambda:list(INPUT.rglob('giada_primitive_scaling_laws.zip'))+[p.parent for p in INPUT.rglob('frozen_scaling_checkpoints.pt')]+[p for p in INPUT.rglob('archive.zip') if 'primitive_scaling' in str(p).lower() or 'primitive-scaling' in str(p).lower()],lambda p,i:verified_task5_any(p,WORK/f'.task5_discovery_{i}'))
override=os.environ.get('GIADA_TARGETED_DATASET');candidates=[Path(override).expanduser()] if override else []
if INPUT.is_dir():candidates += [p.parent for p in INPUT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower()]+[p for p in INPUT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE=next((p.resolve() for p in dict.fromkeys(candidates) if p.exists()),None)
assert BASE is not None,'Dataset targeted v1.1 base non trovato.'
display({'task15c':str(TASK15C),'task5':str(TASK5),'base':str(BASE)})


In [ ]:
def materialize_base(source):
 source=Path(source)
 if source.is_dir():
  assert (source/'dataset_manifest.json').is_file() and (source/'state_schema.json').is_file()
  return source
 destination=WORK/'extracted_base';assert not destination.exists();destination.mkdir(parents=True)
 with zipfile.ZipFile(source) as archive:
  required=('dataset_manifest.json','state_schema.json','transition_dataset.h5')
  members={name:[m for m in archive.infolist() if m.filename.endswith('/'+name)] for name in required}
  assert all(len(items)==1 for items in members.values()),'Archivio base ambiguo o incompleto.'
  assert len({str(Path(items[0].filename).parent) for items in members.values()})==1
  for name in required:
   member=members[name][0];print({'extracting':name,'GiB':round(member.file_size/2**30,3)},flush=True)
   with archive.open(member) as reader,(destination/name).open('xb') as writer:
    copied=0;notice=.1
    while True:
     block=reader.read(16*1024*1024)
     if not block:break
     writer.write(block);copied+=len(block)
     if name.endswith('.h5') and copied/member.file_size>=notice:
      print(f'[GIADA Task 16] estrazione HDF5 {round(100*copied/member.file_size)}%',flush=True);notice+=.1
 return destination
BASE_ROOT=materialize_base(BASE)
print({'base_root':str(BASE_ROOT),'native_snapshots':'solo quelli selezionati verranno copiati nel workspace'})


## 🔬 Replay nativo e candidato shadow
Il codice verifica lo SHA-256 dei 6 GiB, poi campiona soltanto `deterministic_test` secondo il protocollo. Copia solo gli snapshot necessari, ricostruisce la struttura NEURON originale e riproduce ciascuna transizione selezionata. La LUT si aggiorna online nel callback di osservazione a ogni sottopasso; la GPU verifica prima l'equivalenza numerica con il kernel Task5, mentre NEURON e lo shadow procedono sequenzialmente su CPU. Se replay di stato/voltaggio fallisce, salva un report diagnostico e **non** attribuisce il mismatch alla LUT. Il report finale include gate, corrente Ca_HVA calcolata da gate nativi e gate LUT, e risultati per sito/regime. La corrente di riferimento non è `segment.ica`, che somma meccanismi diversi.


In [ ]:
import torch
assert torch.cuda.is_available(),'Task 16 richiede GPU CUDA per la LUT congelata.'
config=FrozenEmbeddedConfig();config.validate()
formula=ExtractedGateFormula.from_mod(TEACHER/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
OUTPUT=Path('/kaggle/working/artifacts/giada_roadmap_task16_embedded_frozen_confirmation')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}. Avvia una sessione pulita.'
def progress(percent,label):print(f'[GIADA Task 16][SHA-256 {label}] {percent}%',flush=True)
report=run_frozen_embedded_confirmation(formula,REPO,TEACHER,BASE_ROOT,BASE,TASK5,TASK15C,OUTPUT,config,code_revision=REVISION,progress=progress)
display({'valid':report['valid'],'diagnosis':report.get('diagnosis'),'support':report['support'],'native_replay_valid':report.get('native_replay_valid'),'floor_valid':report.get('floor_valid'),'gate_valid':report.get('gate_valid'),'current_valid':report.get('current_valid'),'embedded_passed':report.get('embedded_teacher_forced_confirmation_passed'),'gate_c_authorized':report.get('gate_c_authorized',False)})
if not report['valid']:print('NO-GO diagnostico: scarica comunque lo ZIP e non proseguire alla sostituzione causale.')


## 📦 Download compatto
Solo report e indice dei percorsi; gli snapshot e l'HDF5 non entrano nello ZIP. Scarica anche in caso di NO-GO.


In [ ]:
EXPORT=WORK/'download_report';EXPORT.mkdir()
for name in ('final_report.json','selected_test_paths.json'):shutil.copy2(OUTPUT/name,EXPORT/name)
archive=Path(shutil.make_archive('/kaggle/working/giada_roadmap_task16_embedded_frozen_confirmation','zip',EXPORT.parent,EXPORT.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
